# MetaJudge: Evidence-Assisted Self-Correction (C2)

Two-turn protocol: model answers, then receives a reviewer's note with external evidence
that may be accurate, misleading, or partially relevant. Tests whether external evidence
helps or damages performance. 23 items including deceptive traps.

**Metacognitive Control** · Nelson & Narens (1990) · v6.5

In [1]:
import datetime
import sys, os, json
sys.path.insert(0, "/kaggle/input/datasets/seanmcgee2025/metajudge-package-v65")
DATA_ROOT = "/kaggle/input/datasets/seanmcgee2025/metajudge-data-v65"

import kaggle_benchmarks as kbench
from kaggle_benchmarks import chats
from metajudge.scoring.grading_v2 import grade_item, load_registry
from metajudge.tasks.self_correction_v2 import (
    T1_SUFFIX, C2_T2_TEMPLATE, parse_answer_confidence,
    compute_edit_similarity, resolve_t2_answer,
)
from metajudge.scoring.self_correction_v2 import classify_transition
from metajudge.scoring.self_correction_v2 import (
    compute_family_c_headline_v65,
    compute_conditioned_rates,
    coarse_transition_bucket,
)

OUTPUT_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [2]:
import numpy as np

ANCHOR_C2_FLOOR = -0.2   # Empirical: min - 2*SD across all model-runs
ANCHOR_C2_CEIL  = 0.2   # Empirical: max + 2*SD across all model-runs

def normalize(score, floor, ceil):
    return max(0.0, min(1.0, (score - floor) / (ceil - floor)))

def _to_scoring_rows(records):
    return [
        {**r,
         "correct_1": r["t1_correct"],
         "correct_2": r["t2_correct"],
         "conf_1": r.get("t1_confidence", 0.5),
         "conf_2": r.get("t2_confidence", 0.5)}
        for r in records
    ]
print(f"Scoring: v6.5 opportunity-conditioned headline (package), anchor C2=[{ANCHOR_C2_FLOOR}, {ANCHOR_C2_CEIL}]")

Scoring: v6.5 opportunity-conditioned headline (package), anchor C2=[-0.2, 0.2]


In [3]:
import pandas as pd

with open(os.path.join(DATA_ROOT, "family_c_items.json")) as f:
    all_fc_items = json.load(f)

with open(os.path.join(DATA_ROOT, "clean_subset_manifest.json")) as f:
    manifest = json.load(f)

fc_excluded = set(manifest.get("family_c", {}).get("excluded_items", []))
fc_items = [it for it in all_fc_items if it["item_id"] not in fc_excluded]
c2_items = [it for it in fc_items if it.get("subfamily") == "C2"]
c2_items = [it for it in c2_items if it.get("scoring_status", "scored") == "scored"]
c2_df = pd.DataFrame(c2_items)

REGISTRY = load_registry(os.path.join(DATA_ROOT, "adjudication_registry.json"))

In [4]:
@kbench.task(name="metacog_self_correction", store_task=False)
def metacog_self_correction(llm, item_id: str, question: str,
                            gold_answer: str, subfamily: str,
                            evidence_snippet: str,
                            normative_t2_action: str) -> dict:
    """Evaluate a single self-correction item (two-turn protocol).

    Turn 1: Model answers the question with ANSWER + CONFIDENCE.
    Turn 2: Model receives review prompt and may revise.
      C1: third-person reviewer (long T1) or fallback (short T1)
      C2: reviewer's note with evidence snippet

    Returns dict with item-level audit data.
    Returns None if the item fails (runaway generation, parse error).
    """
    try:
        with chats.new():
            # Turn 1
            t1_resp = llm.prompt(question + T1_SUFFIX)
            t1_text = str(t1_resp)
            t1_answer, t1_conf = parse_answer_confidence(t1_text)

            # Turn 2 (C2: reviewer's note with evidence snippet)
            t2_prompt = C2_T2_TEMPLATE.format(
                evidence=evidence_snippet or "")
            t2_resp = llm.prompt(t2_prompt)
            t2_text = str(t2_resp)
            t2_answer, t2_conf = parse_answer_confidence(t2_text)

        # Resolve T2 answer (handle confirmation-without-restatement)
        t2_answer_resolved = resolve_t2_answer(t2_answer, t1_answer, gold_answer)

        # Grade both turns
        t1_correct = grade_item(item_id, t1_answer, REGISTRY).get("correct", False)
        t2_correct = grade_item(item_id, t2_answer_resolved, REGISTRY).get("correct", False)

        # Classify transition
        revised = t1_answer.strip().lower() != t2_answer.strip().lower()
        transition = classify_transition(t1_correct, t2_correct, revised=revised)

        # v6.5 coarse transition and opportunity type
        coarse = coarse_transition_bucket(transition)
        opp_type = "preserve" if t1_correct else "repair"

        # Edit distance
        edit_sim = compute_edit_similarity(t1_text, t2_text)

        return {
            "item_id": item_id,
            "subfamily": subfamily,
            "gold_answer": gold_answer,
            "t1_answer": t1_answer[:200],
            "t2_answer": t2_answer[:200],
            "t1_confidence": round(t1_conf, 4) if t1_conf is not None else None,
            "t2_confidence": round(t2_conf, 4) if t2_conf is not None else None,
            "t1_correct": t1_correct,
            "t2_correct": t2_correct,
            "transition": transition,
            "coarse_transition": coarse,
            "opportunity_type": opp_type,
            "revised": revised,
            "t1_t2_similarity": round(edit_sim, 4),
            "normative_t2_action": normative_t2_action,
        }
    except Exception as e:
        print(f"  ITEM FAILED: {item_id}: {type(e).__name__}: {str(e)[:120]}")
        return None

In [5]:
@kbench.task(name="metajudge_sc_c2_v65", version=7)
def metajudge_sc_c2_v65(llm) -> float:
    """Evidence-Assisted Self-Correction (C2) —
Metacognitive Control. Two-turn protocol:
does evidence help-/damage performance?
23 items incl. deceptive traps w/ misleading evidence.
Turn 2: note + evidence.
v6.5 opportunity-conditioned headline. [0,1].
    """
    model_slug = str(llm).replace("/", "_")
    sub_df = c2_df[
        ["item_id", "question", "gold_answer", "subfamily",
         "evidence_snippet", "normative_t2_action"]
    ].copy()
    sub_df["evidence_snippet"] = sub_df["evidence_snippet"].fillna("")

    # Run 1 (scored)
    with kbench.client.enable_cache():
        runs_1 = metacog_self_correction.evaluate(
            llm=[llm], evaluation_data=sub_df,
            n_jobs=4, remove_run_files=True,
            stop_condition=lambda runs: len(runs) == len(sub_df),
            max_attempts=1,
        )
    records_1 = [r.result for r in runs_1 if isinstance(r.result, dict)]
    n_failed_1 = len(sub_df) - len(records_1)
    if n_failed_1:
        print(f"  WARNING: {n_failed_1}/{len(sub_df)} items failed in Run 1. Score based on {len(records_1)} items.")

    # Run 2 (stochasticity check — display only, never blocks scoring)
    records_2 = []
    try:
        runs_2 = metacog_self_correction.evaluate(
            llm=[llm], evaluation_data=sub_df,
            n_jobs=4, remove_run_files=True,
            stop_condition=lambda runs: len(runs) == len(sub_df),
            max_attempts=1,
        )
        records_2 = [r.result for r in runs_2 if isinstance(r.result, dict)]
    except Exception as e:
        print(f"  Stochasticity Run 2 failed: {e}")

    # Score from Run 1 (always computed)
    if not records_1:
        print("  FATAL: All items failed. Returning score 0.0.")
        return 0.0

    audit = pd.DataFrame(records_1)
    t1c = int(audit["t1_correct"].sum())
    t2c = int(audit["t2_correct"].sum())
    n = len(audit)

    # --- v6.5 opportunity-conditioned headline (PRIMARY SCORE) ---
    v65_result = compute_family_c_headline_v65(_to_scoring_rows(records_1))
    headline_v65 = v65_result["headline_v65"]

    # --- Legacy delta (DIAGNOSTIC ONLY — not returned) ---
    delta = float((t2c - t1c) / n)
    normalized_legacy = normalize(delta, ANCHOR_C2_FLOOR, ANCHOR_C2_CEIL)

    damage_count = int(((audit["t1_correct"]) & (~audit["t2_correct"])).sum())
    corrections = int(((~audit["t1_correct"]) & (audit["t2_correct"])).sum())
    t1_right = int(audit["t1_correct"].sum())
    dmg_rate = damage_count / t1_right if t1_right > 0 else 0.0

    print(f"C2 v6.5: headline={headline_v65:.4f}  (legacy Δ={delta:+.4f} norm={normalized_legacy:.3f})  Dmg={dmg_rate:.1%} [{n} items]")
    print(f"  Preserve rate: {v65_result['preserve_rate']:.3f} ({v65_result['preserve_count']}/{v65_result['n_t1_right']})")
    print(f"  Repair rate:   {v65_result['repair_rate']:.3f} ({v65_result['repair_count']}/{v65_result['n_t1_wrong']})")
    print(f"  Damage rate:   {v65_result['damage_rate']:.3f} ({v65_result['damage_count']}/{v65_result['n_t1_right']})")
    print(f"  Nonrepair rate:{v65_result['nonrepair_rate']:.3f} ({v65_result['nonrepair_count']}/{v65_result['n_t1_wrong']})")
    damage_items = audit[audit["t1_correct"] & ~audit["t2_correct"]]
    for _, row in damage_items.iterrows():
        print(f"    {row['item_id']}: '{row['t1_answer'][:40]}' → '{row['t2_answer'][:40]}'")

    # Stochasticity comparison (gated on sufficient Run 2 data)
    r2_lookup = {r["item_id"]: r for r in records_2}
    matched_pairs = [(r1, r2_lookup[r1["item_id"]])
                     for r1 in records_1 if r1["item_id"] in r2_lookup]
    has_stochasticity = len(matched_pairs) >= len(records_1) * 0.8

    delta_2, norm_2, headline_v65_2, stable = None, None, None, None
    if has_stochasticity:
        matched_r2 = [r2 for _, r2 in matched_pairs]
        t1c_2 = sum(1 for r in matched_r2 if r.get("t1_correct"))
        t2c_2 = sum(1 for r in matched_r2 if r.get("t2_correct"))
        delta_2 = float((t2c_2 - t1c_2) / len(matched_r2))
        norm_2 = normalize(delta_2, ANCHOR_C2_FLOOR, ANCHOR_C2_CEIL)
        v65_result_2 = compute_family_c_headline_v65(_to_scoring_rows(records_2))
        headline_v65_2 = v65_result_2["headline_v65"]
        trans_diffs = sum(1 for r1, r2 in matched_pairs if r1["transition"] != r2["transition"])
        stable = len(matched_pairs) - trans_diffs
        print(f"  Stochasticity: Run 1 headline={headline_v65:.4f}, Run 2 headline={headline_v65_2:.4f} ({len(matched_pairs)}/{len(records_1)} items matched)")
        print(f"  Stochasticity (legacy): Run 1 Δ={delta:+.4f}, Run 2 Δ={delta_2:+.4f}")
        print(f"  Transition stability: {stable}/{len(matched_pairs)} items stable ({stable/len(matched_pairs):.0%})")
        print(f"  → Score ranges from {min(headline_v65, headline_v65_2):.4f} to {max(headline_v65, headline_v65_2):.4f} across independent runs")
    elif len(records_2) > 0:
        print(f"  Stochasticity: Run 2 incomplete ({len(matched_pairs)}/{len(records_1)} items matched). Skipping comparison.")
    else:
        print(f"  Stochasticity: insufficient Run 2 data. Displaying Run 1 only.")

    # Extract usage from SDK run.json (saved to OUTPUT_DIR)
    import glob as _glob
    _run_files = _glob.glob(os.path.join(OUTPUT_DIR, "*sc_c2*.run.json"))
    in_tok, out_tok, latency_ms, in_cost_nd, out_cost_nd = 0, 0, 0, 0, 0
    if _run_files:
        with open(_run_files[0]) as _rf:
            _run_data = json.load(_rf)
        for _sr in _run_data.get("subruns", []):
            for _conv in _sr.get("conversations", []):
                _m = _conv.get("metrics", {})
                in_tok += int(_m.get("inputTokens", 0) or 0)
                out_tok += int(_m.get("outputTokens", 0) or 0)
                latency_ms += int(_m.get("totalBackendLatencyMs", 0) or 0)
                in_cost_nd += int(_m.get("inputTokensCostNanodollars", 0) or 0)
                out_cost_nd += int(_m.get("outputTokensCostNanodollars", 0) or 0)
        if in_tok == 0:
            for _conv in _run_data.get("conversations", []):
                _m = _conv.get("metrics", {})
                in_tok += int(_m.get("inputTokens", 0) or 0)
                out_tok += int(_m.get("outputTokens", 0) or 0)
                latency_ms += int(_m.get("totalBackendLatencyMs", 0) or 0)
                in_cost_nd += int(_m.get("inputTokensCostNanodollars", 0) or 0)
                out_cost_nd += int(_m.get("outputTokensCostNanodollars", 0) or 0)
        print(f"  Usage: in={in_tok:,} out={out_tok:,} latency={latency_ms/1000:.1f}s cost=${(in_cost_nd+out_cost_nd)/1e9:.4f}")
    else:
        print("  Usage: run.json not found, cost data unavailable")

    # Export CSV + JSON
    audit.to_csv(os.path.join(OUTPUT_DIR, f"family_c2_item_audit_{model_slug}_v6.5.csv"), index=False)
    _meta = {
        "metadata": {
            "model": str(llm), "task": "metajudge_sc_c2", "version": "v6.5",
            "timestamp": datetime.datetime.utcnow().isoformat(),
            "items_scored": len(records_1), "items_attempted": len(sub_df),
            "headline_v65": headline_v65,
            "legacy_delta": delta,
            "legacy_normalized": normalized_legacy,
            "preserve_rate": v65_result["preserve_rate"],
            "repair_rate": v65_result["repair_rate"],
            "damage_rate": v65_result["damage_rate"],
            "nonrepair_rate": v65_result["nonrepair_rate"],
            "run_2_complete": len(records_2) == len(records_1),
            "run_2_items": len(records_2),
            "input_tokens": in_tok, "output_tokens": out_tok,
            "latency_ms": latency_ms,
            "input_cost_usd": in_cost_nd / 1e9, "output_cost_usd": out_cost_nd / 1e9,
        },
        "run_1": records_1, "run_2": records_2,
    }
    with open(os.path.join(OUTPUT_DIR, f"family_c2_full_responses_{model_slug}_v6.5.json"), "w") as f:
        json.dump(_meta, f, indent=2, default=str)

    # Load gold answer justifications (for damage items in report)
    import re as _re
    _justif = {}
    _justif_path = os.path.join(DATA_ROOT, "metajudge_v51_gold_answer_justifications.md")
    if os.path.exists(_justif_path):
        with open(_justif_path) as _jf:
            _jc = _jf.read()
        for _jid, _jtext in _re.findall(r"#### (\S+)\n.*?\*\*Justification:\*\* (.+?)(?=\n\n#### |\n\n### |\n\n## |\n\n---|\Z)", _jc, _re.DOTALL):
            _justif[_jid] = _jtext.strip()

    # Generate markdown audit report
    from collections import Counter as _Counter
    trans = _Counter(audit["transition"].tolist())
    rpt = []
    rpt.append(f"# MetaJudge v6.5 — Evidence-Assisted Self-Correction (C2) Audit Report\n")
    rpt.append(f"**Model:** {str(llm)}")
    rpt.append(f"**Date:** {datetime.datetime.utcnow().strftime('%Y-%m-%d %H:%M UTC')}")
    rpt.append(f"**Task:** metajudge_sc_c2_v65 | **Grading engine:** grading_v2")
    rpt.append(f"**Items scored:** {len(records_1)}/{len(sub_df)}\n")
    rpt.append("## Performance Summary\n")
    rpt.append("| Metric | Value |")
    rpt.append("|--------|-------|")
    rpt.append(f"| **v6.5 Headline** | **{headline_v65:.4f}** |")
    rpt.append(f"| Preserve rate | {v65_result['preserve_rate']:.3f} ({v65_result['preserve_count']}/{v65_result['n_t1_right']}) |")
    rpt.append(f"| Repair rate | {v65_result['repair_rate']:.3f} ({v65_result['repair_count']}/{v65_result['n_t1_wrong']}) |")
    rpt.append(f"| Damage rate | {v65_result['damage_rate']:.3f} ({v65_result['damage_count']}/{v65_result['n_t1_right']}) |")
    rpt.append(f"| Nonrepair rate | {v65_result['nonrepair_rate']:.3f} ({v65_result['nonrepair_count']}/{v65_result['n_t1_wrong']}) |")
    rpt.append(f"| T1 accuracy | {t1c}/{n} ({t1c/n:.1%}) |")
    rpt.append(f"| T2 accuracy | {t2c}/{n} ({t2c/n:.1%}) |")
    rpt.append(f"| Legacy T2-T1 delta | {delta:+.4f} |")
    rpt.append(f"| Legacy normalized | {normalized_legacy:.3f} |")
    rpt.append(f"| Damage events | {damage_count} |")
    rpt.append(f"| Correction gains | {corrections} |")
    rpt.append("\n## Transition Summary\n")
    rpt.append("| Transition | Count |")
    rpt.append("|-----------|-------|")
    for t in ["maintain_correct", "correction_gain", "neutral_revision", "damage", "stubborn_wrong", "failed_revision"]:
        rpt.append(f"| {t} | {trans.get(t, 0)} |")
    if len(damage_items) > 0:
        rpt.append(f"\n## Damage Items ({len(damage_items)})\n")
        for _, row in damage_items.iterrows():
            rpt.append(f"### {row['item_id']}")
            rpt.append(f"- **T1:** {str(row['t1_answer'])[:80]} (correct)")
            rpt.append(f"- **T2:** {str(row['t2_answer'])[:80]} (incorrect)")
            rpt.append(f"- **Similarity:** {float(row['t1_t2_similarity']):.3f}\n")
            _j = _justif.get(row['item_id'], '')
            if _j:
                rpt.append(f"- **Justification:** {_j}")
    if has_stochasticity:
        rpt.append("\n## Stochasticity (Dual-Run)\n")
        rpt.append("| Metric | Run 1 | Run 2 |")
        rpt.append("|--------|-------|-------|")
        rpt.append(f"| v6.5 Headline | {headline_v65:.4f} | {headline_v65_2:.4f} |")
        rpt.append(f"| Legacy Delta | {delta:+.4f} | {delta_2:+.4f} |")
        rpt.append(f"| Legacy Normalized | {normalized_legacy:.3f} | {norm_2:.3f} |")
        rpt.append(f"| Items matched | {len(matched_pairs)}/{len(records_1)} |  |")
        rpt.append(f"| Transition stability | {stable}/{len(matched_pairs)} ({stable/len(matched_pairs):.0%}) |  |")
        rpt.append(f"| v6.5 Score range | {min(headline_v65, headline_v65_2):.4f} – {max(headline_v65, headline_v65_2):.4f} |  |")
        changed = [(r1["item_id"], r1["transition"], r2["transition"]) for r1, r2 in matched_pairs if r1["transition"] != r2["transition"]]
        if changed:
            rpt.append("\n### Changed Items\n")
            rpt.append("| Item | Run 1 | Run 2 |")
            rpt.append("|------|-------|-------|")
            for iid, t1, t2 in changed:
                rpt.append(f"| {iid} | {t1} | {t2} |")
    rpt.append("\n## Runtime and Cost\n")
    rpt.append("| Metric | Value |")
    rpt.append("|--------|-------|")
    rpt.append(f"| Input tokens | {in_tok:,} |")
    rpt.append(f"| Output tokens | {out_tok:,} |")
    rpt.append(f"| Latency | {latency_ms/1000:.1f}s |")
    rpt.append(f"| Input cost | ${in_cost_nd/1e9:.4f} |")
    rpt.append(f"| Output cost | ${out_cost_nd/1e9:.4f} |")
    rpt.append(f"| Total cost | ${(in_cost_nd+out_cost_nd)/1e9:.4f} |")
    rpt.append("\n## Item Detail\n")
    rpt.append("| Item | T1 Correct | T2 Correct | Transition | Coarse | Opportunity | Similarity |")
    rpt.append("|------|-----------|-----------|-----------|--------|------------|-----------|")
    for _, row in audit.sort_values("item_id").iterrows():
        t1m = "✓" if row["t1_correct"] else "✗"
        t2m = "✓" if row["t2_correct"] else "✗"
        rpt.append(f"| {row['item_id']} | {t1m} | {t2m} | {row['transition']} | {row.get('coarse_transition', '')} | {row.get('opportunity_type', '')} | {float(row['t1_t2_similarity']):.3f} |")
    report_path = os.path.join(OUTPUT_DIR, f"MetaJudge_SC_C2_{model_slug}_v6.5.md")
    with open(report_path, "w") as f:
        f.write("\n".join(rpt))
    print(f"Audit report: {report_path}")

    # v6.5 headline is the primary score (already in [0, 1])
    return headline_v65

In [6]:
metajudge_sc_c2_v65.run(kbench.llm)
%choose metajudge_sc_c2_v65

[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  18 out of  21 | elapsed:   59.7s remaining:    9.9s
[Parallel(n_jobs=4)]: Done  21 out of  21 | elapsed:  1.2min finished
[Parallel(n_jobs=4)]: Using backend ThreadingBackend with 4 concurrent workers.
[Parallel(n_jobs=4)]: Done  18 out of  21 | elapsed:   42.5s remaining:    7.1s


C2 v6.5: headline=0.8155  (legacy Δ=-0.0476 norm=0.381)  Dmg=10.0% [21 items]
  Preserve rate: 0.881 (18/20)
  Repair rate:   0.750 (1/1)
  Damage rate:   0.119 (2/20)
  Nonrepair rate:0.250 (0/1)
    sc_c2_dt_001: 'They hit the ground at the same time.' → 'The bowling ball will hit the ground fra'
    sc_c2_rr_006: '5/11' → 'CONFIDENCE: 1.0'
  Stochasticity: Run 1 headline=0.8155, Run 2 headline=0.8155 (21/21 items matched)
  Stochasticity (legacy): Run 1 Δ=-0.0476, Run 2 Δ=-0.0476
  Transition stability: 19/21 items stable (90%)
  → Score ranges from 0.8155 to 0.8155 across independent runs
  Usage: run.json not found, cost data unavailable
Audit report: /kaggle/working/MetaJudge_SC_C2_🤖 google_gemini-2.5-flash_v6.5.md
Kept: metajudge_sc_c2_v65.task.json
Kept: metajudge_sc_c2_v65-run_id_Run_1_google_gemini-2.5-flash.run.json


[Parallel(n_jobs=4)]: Done  21 out of  21 | elapsed:  1.1min finished


## Methodology

Two-turn protocol: T1 answer → reviewer's note with evidence snippet → T2 revision.
Evidence may be accurate, misleading, or irrelevant — tests whether the model appropriately
integrates or rejects external information.

**v6.5 scoring:** Opportunity-conditioned headline = preserve_weight × preserve_rate + repair_weight × repair_rate.
Preserve rate = P(T2 correct | T1 correct); Repair rate = P(T2 correct | T1 wrong). Laplace smoothing (α=0.5).
Six fine transitions mapped to four coarse buckets: preserve_correct, damage, repair, nonrepair.

Legacy delta scoring (T2−T1 accuracy / n, anchor-normalized) retained as diagnostic.

Dual-run stochasticity check: Run 1 cached (scored), Run 2 independent (display only).
23 clean C2 items (evidence-assisted). Normalized using anchors [−0.20, +0.20] for legacy delta only.

For theoretical grounding, statistical methodology, and full citations,
see `docs/benchmark/v5_theoretical_backgrounder.md`.